# ShopSphere — Enterprise Data Quality Assessment & Profiling
**Project**: ShopSphere E-commerce Sales, Customer & Business Performance Analytics  
**Role**: Data Analyst + Business Analyst  
**Objective**: Audit the 5 raw ingestion feeds (`customers`, `products`, `orders`, `delivery`, `returns`), profile missing values, detect duplicates, uncover formatting inconsistencies, flag outliers, and evaluate referential integrity before downstream ingestion.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 5)

RAW_DIR = os.path.join("..", "data", "raw")

# Load raw operational tables
df_c_raw = pd.read_csv(os.path.join(RAW_DIR, "customers.csv"))
df_p_raw = pd.read_csv(os.path.join(RAW_DIR, "products.csv"))
df_o_raw = pd.read_csv(os.path.join(RAW_DIR, "orders.csv"))
df_d_raw = pd.read_csv(os.path.join(RAW_DIR, "delivery.csv"))
df_r_raw = pd.read_csv(os.path.join(RAW_DIR, "returns.csv"))

print("Raw datasets loaded successfully into memory.")

## 1. Shape Analysis & High-Level Ingestion Volume
We begin by assessing the dimensional cardinality and column count across each raw ingestion feed.

In [ ]:
shapes = {
    "Customers": df_c_raw.shape,
    "Products": df_p_raw.shape,
    "Orders": df_o_raw.shape,
    "Delivery": df_d_raw.shape,
    "Returns": df_r_raw.shape
}
df_shapes = pd.DataFrame(shapes, index=["Row Count", "Column Count"]).T
df_shapes["Memory (MB)"] = [
    df_c_raw.memory_usage().sum() / 1e6,
    df_p_raw.memory_usage().sum() / 1e6,
    df_o_raw.memory_usage().sum() / 1e6,
    df_d_raw.memory_usage().sum() / 1e6,
    df_r_raw.memory_usage().sum() / 1e6
]
df_shapes

## 2. Missing Value Analysis
Missingness assessment reveals non-random missing patterns in demographic fields (`gender`, `city`), operational fields (`brand`), and fulfillment tracking attributes (`delivery_status`, `return_reason`).

In [ ]:
def profile_missing(df, name):
    missing = df.isna().sum()
    pct = (missing / len(df)) * 100
    res = pd.DataFrame({"Missing Records": missing, "Missing Pct (%)": pct})
    res = res[res["Missing Records"] > 0]
    if len(res) == 0:
        return f"{name}: No missing values detected."
    print(f"--- Missing Values: {name} ---")
    return res

print(profile_missing(df_c_raw, "Customers"))
print(profile_missing(df_p_raw, "Products"))
print(profile_missing(df_o_raw, "Orders"))
print(profile_missing(df_d_raw, "Delivery"))
print(profile_missing(df_r_raw, "Returns"))

## 3. Duplicate Records Detection
In transactional systems, duplicate primary keys lead to inflated revenue and double-counted operational metrics.

In [ ]:
dups_summary = {
    "Table": ["Customers", "Products", "Orders", "Delivery", "Returns"],
    "Primary Key": ["customer_id", "product_id", "order_id", "order_id", "return_id"],
    "Duplicate PKs": [
        df_c_raw.duplicated(subset=["customer_id"]).sum(),
        df_p_raw.duplicated(subset=["product_id"]).sum(),
        df_o_raw.duplicated(subset=["order_id"]).sum(),
        df_d_raw.duplicated(subset=["order_id"]).sum(),
        df_r_raw.duplicated(subset=["return_id"]).sum()
    ],
    "Exact Row Duplicates": [
        df_c_raw.duplicated().sum(),
        df_p_raw.duplicated().sum(),
        df_o_raw.duplicated().sum(),
        df_d_raw.duplicated().sum(),
        df_r_raw.duplicated().sum()
    ]
}
pd.DataFrame(dups_summary)

## 4. Referential Integrity & Foreign Key Validation
Verifying whether all orders resolve cleanly to valid customers and products, and whether returns bind to known orders.

In [ ]:
valid_custs = set(df_c_raw["customer_id"].dropna())
valid_prods = set(df_p_raw["product_id"].dropna())
valid_orders = set(df_o_raw["order_id"].dropna())

orphaned_cust_orders = (~df_o_raw["customer_id"].isin(valid_custs)).sum()
orphaned_prod_orders = (~df_o_raw["product_id"].isin(valid_prods)).sum()
orphaned_returns = (~df_r_raw["order_id"].isin(valid_orders)).sum()

pd.DataFrame({
    "Referential Relationship": [
        "Orders -> Customers (customer_id)",
        "Orders -> Products (product_id)",
        "Returns -> Orders (order_id)"
    ],
    "Unlinked / Orphaned Records": [orphaned_cust_orders, orphaned_prod_orders, orphaned_returns]
})

## 5. Domain Boundary & Outlier Detection
Examining categorical casing drift, negative numeric values, and biological/operational outliers.

In [ ]:
print("Customer Segment Casing Distribution:")
print(df_c_raw["customer_segment"].value_counts(dropna=False))

print("\nProduct Category Casing Distribution:")
print(df_p_raw["category"].value_counts(dropna=False))

print("\nCustomer Age Outliers (<= 0 or > 100):")
print(df_c_raw[(df_c_raw["age"] <= 0) | (df_c_raw["age"] > 100)][["customer_id", "customer_name", "age"]].head(10))

print("\nOrder Quantity Anomalies (<= 0 or > 50):")
print(df_o_raw[(df_o_raw["quantity"] <= 0) | (df_o_raw["quantity"] > 50)][["order_id", "product_id", "quantity", "discount"]].head(10))

## 6. Summary of Data Quality Findings
1. **Deduplication Required**: Across all tables, primary keys contain intentional ingestion duplicates that must be pruned.
2. **Standardization Required**: String casing anomalies in `customer_segment` and `category` require normalization.
3. **Imputation & Cleansing**: Demographic missingness (`gender`, `city`) must be filled with standard placeholders, while age outliers must be imputed with median.
4. **Referential Pruning**: Orders lacking valid foreign keys cannot be attributed to customers/products and must be excluded from analytical models.